In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Data
df = pd.read_csv('Dataset.csv')

# 2. Preprocess Text Data
df['Cuisines'] = df['Cuisines'].fillna('Unknown')

# 3. TF-IDF Matrix on Cuisines
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Cuisines'])

# 4. Cosine Similarity Compute
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 5. Recommendation Function
indices = pd.Series(df.index, index=df['Restaurant Name']).drop_duplicates()

def get_recommendations(restaurant_name, top_n=5):
    if restaurant_name not in indices:
        return f"Restaurant '{restaurant_name}' not found."

    idx = indices[restaurant_name]
    # Agar multiple restaurants same name ke ho to pehla index lo
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    rest_indices = [i[0] for i in sim_scores]

    return df[['Restaurant Name', 'Cuisines', 'Aggregate rating', 'Average Cost for two']].iloc[rest_indices]

# Test recommendation system
sample_restaurant = df['Restaurant Name'].iloc[0]
print(f"Top 5 Recommendations for '{sample_restaurant}':\n")
print(get_recommendations(sample_restaurant))

Top 5 Recommendations for 'Le Petit Souffle':

            Restaurant Name          Cuisines  Aggregate rating  \
4993        Tokyo Mon Amour  Japanese, French               3.1   
70        Paris 6 Classique            French               3.4   
259                  Django            French               4.3   
460                    Jaan            French               3.8   
461   Rhubarb Le Restaurant            French               3.9   

      Average Cost for two  
4993                  2200  
70                     200  
259                     40  
460                    430  
461                    315  
